# 22 — GROVER Embeddings (tencent-ailab/grover)
Self-supervised graph neural network pretrained on 10M unlabeled molecules
using contextual property prediction. Produces graph-level fingerprints by
reading out atom/bond representations from a dual message-passing transformer.
Unlike SMILES-sequence models, GROVER operates directly on molecular graphs.
Runtime: ~2-3h (weight download ~900MB + inference).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings, os
sys.path.insert(0, "../src")
sys.path.insert(0, "../checkpoints/grover_repo")  # add GROVER to path
warnings.filterwarnings("ignore")

from pathlib import Path
import csv
import tarfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm.auto import tqdm

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
GROVER_REPO    = Path("../checkpoints/grover_repo")
GROVER_WEIGHTS = Path("../checkpoints/grover_base.pt")
GROVER_BASE_URL = "https://drive.google.com/file/d/1hiGwOzoRfbJQPWj0V_mtOffsqIIAMgjl/view?usp=drive_link"
GDRIVE_FILE_ID  = "1hiGwOzoRfbJQPWj0V_mtOffsqIIAMgjl"

print(f"GROVER repo exists:    {GROVER_REPO.exists()}")
print(f"GROVER weights exist:  {GROVER_WEIGHTS.exists()}")

In [ ]:
# ── 2. Download GROVER pretrained weights if needed ──────────────────────────
if not GROVER_WEIGHTS.exists():
    print("Downloading GROVER base pretrained weights via gdown (~900MB)...")
    import gdown
    # Downloads grover_base.tar.gz, extracts to get grover_base.pt
    tar_path = Path("../checkpoints/grover_base.tar.gz")
    gdown.download(id=GDRIVE_FILE_ID, output=str(tar_path), quiet=False)
    if tar_path.exists():
        print(f"Downloaded: {tar_path}  ({tar_path.stat().st_size / 1e6:.1f} MB)")
        with tarfile.open(str(tar_path)) as tf:
            tf.extractall("../checkpoints/")
        print("Extracted successfully.")
        # Find the .pt file
        pt_files = list(Path("../checkpoints/").glob("**/*.pt"))
        print(f"Found .pt files: {[str(p) for p in pt_files]}")
        if pt_files:
            GROVER_WEIGHTS = pt_files[0]
    else:
        raise FileNotFoundError(
            "gdown download failed. Download manually from Google Drive and "
            "place at checkpoints/grover_base.pt"
        )
else:
    print(f"Weights already present: {GROVER_WEIGHTS}  "
          f"({GROVER_WEIGHTS.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# ── 3. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()

smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

print(f"Train: {len(smiles_tr):,}  |  Test: {len(smiles_te):,}")
print(f"y_tr range: {y_tr.min():.3f} – {y_tr.max():.3f}")

In [ ]:
# ── 4. Extract GROVER fingerprints ──────────────────────────────────────────
import torch
from argparse import Namespace

# Import GROVER modules (repo already on sys.path)
from grover.util.utils import load_checkpoint, create_logger, get_data
from grover.data import MoleculeDataset, MolCollator
from torch.utils.data import DataLoader

CACHE_TR = DATA_PROCESSED / 'grover_train_emb.npy'
CACHE_TE = DATA_PROCESSED / 'grover_test_emb.npy'


def write_smiles_csv(smiles_list: list, path: Path):
    """Write SMILES to a single-column CSV (no header) for GROVER's get_data."""
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        for smi in smiles_list:
            w.writerow([smi])


def make_grover_args(weights_path: Path, data_path: Path) -> Namespace:
    return Namespace(
        parser_name='fingerprint',
        fingerprint_source='atom',   # outputs 2*hidden_size
        checkpoint_paths=[str(weights_path)],
        data_path=str(data_path),
        features_generator=None,
        features_path=None,
        features_dim=0,
        max_data_size=float('inf'),
        use_compound_names=False,
        skip_invalid_smiles=False,
        no_cache=True,
        batch_size=32,
        cuda=False,
        bond_drop_rate=0,
        attn_out=4,
        # These get overridden from checkpoint:
        hidden_size=300,
        depth=6,
        heads=4,
        ffn_num_layers=2,
        ffn_hidden_size=300,
        dropout=0.0,
        activation='ReLU',
        undirected=False,
        dense=False,
        self_attention=False,
        aug_rate=0,
        dist_coff=0,
    )


def extract_grover_fps(smiles_list: list, cache: Path,
                       weights_path: Path, tmp_csv: Path) -> np.ndarray:
    if cache.exists():
        print(f"  Loading cached {cache.name}")
        return np.load(str(cache))
    write_smiles_csv(smiles_list, tmp_csv)
    args = make_grover_args(weights_path, tmp_csv)
    logger = create_logger('grover_fp', save_dir=None, quiet=True)
    print("  Loading GROVER checkpoint ...")
    model = load_checkpoint(str(weights_path), current_args=args, cuda=False, logger=logger)
    model.eval()
    print("  Running fingerprint generation ...")
    from task.fingerprint import do_generate
    test_data = get_data(
        path=str(tmp_csv), args=args,
        use_compound_names=False,
        max_data_size=float('inf'),
        skip_invalid_smiles=False
    )
    test_data = MoleculeDataset(test_data)
    fps = do_generate(model, test_data, args)
    X = np.array(fps, dtype=np.float32)
    np.save(str(cache), X)
    tmp_csv.unlink(missing_ok=True)
    return X


TMP_TR = DATA_PROCESSED / '_grover_train_tmp.csv'
TMP_TE = DATA_PROCESSED / '_grover_test_tmp.csv'

print("Extracting train GROVER embeddings ...")
X_tr = extract_grover_fps(smiles_tr, CACHE_TR, GROVER_WEIGHTS, TMP_TR)
print(f"  Train: {X_tr.shape}")
print("Extracting test GROVER embeddings ...")
X_te = extract_grover_fps(smiles_te, CACHE_TE, GROVER_WEIGHTS, TMP_TE)
print(f"  Test:  {X_te.shape}")
print(f"  NaN: train={np.isnan(X_tr).sum()}  test={np.isnan(X_te).sum()}")

In [ ]:
# ── 5. Scaffold 5-fold CV with LGBM OOF ───────────────────────────────────────
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof[va_idx] = m.predict(X_tr[va_idx])
    fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    met['fold'] = fold_i
    fold_metrics.append(met)
    print(f"  Fold {fold_i + 1}: RAE={fold_rae:.4f}  Spearman={met['Spearman']:.4f}")

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f"\nOOF RAE (global): {oof_rae:.4f}")
print(f"Mean fold RAE:    {cv_df['RAE'].mean():.4f} +/- {cv_df['RAE'].std():.4f}")
print()
print("== Comparison ==")
print(f"  LGBM_base (Morgan + RDKit only): ~0.575")
print(f"  Chemprop multitask (nb 03):       0.517")
print(f"  ChemBERTa-zinc-MLM (nb 13):       0.6782")
print(f"  ChemBERTa-PubChem-MTR (nb 14):    0.5993")
print(f"  Grand ensemble best:              0.5363")
print(f"  GROVER (this nb):                {oof_rae:.4f}")

np.save(DATA_PROCESSED / 'oof_grover.npy', oof)

In [ ]:
# ── 6. Full retrain on all train data + predict test ──────────────────────────
final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_tr, y_tr)
te_preds = final_m.predict(X_te)
te_preds = np.clip(te_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_grover.npy', te_preds)
print(f"Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}")
print(f"Test preds range: {te_preds.min():.3f} – {te_preds.max():.3f}")

In [ ]:
# ── 7. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds
})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '22_grover.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"OOF RAE (GROVER LGBM): {oof_rae:.4f}")
print(sub['pEC50'].describe().round(3))